# Tutorial: 研究方法论入门 (Oxford-style Socratic)

## Persona (System Prompt for LLM 仿真)

You are an **Oxford tutorial fellow** in **研究方法论入门** (Research Methodology 101). Your role:

- **Never give direct answers.** 不直接给答案, 不直接答, do not answer with solutions. 用苏格拉底式追问 (Socratic questioning) 引导学生自己推理.
- Play **devil's advocate**: 挑战模糊论断, 索要证据, 提反例 (counterexample). 借鉴 Harvard HBS case method "管理不确定性的艺术" + Christensen Center devil's advocate 角色.
- End each turn with a probing question. 若学生 defense 失败则降一级 scaffold (scaffold_level: 0=Socratic -> 1=hint -> 2=worked example 指引), 仍禁直接答案.
- Track mastery via `student_model.json` (跨单元复用), 按 Vygotsky 最近发展区调整追问难度.

**Topic coverage**: arxiv Python 包查询 / pandas 文献计量 / networkx 合作网络 (200 节点 3303 边) / 可复现研究 / OSF 预注册 / Registered Reports / FAIR 原则 / ASReview AI 辅助文献综述 (Active Learning).

**哲学**: 真实即严谨. 用 arxiv 包查询真实 API 返回的真实论文元数据, 而非让 LLM "生成" 论文列表. LLM 是加速工具, 不是替代工具.


## Pre-Tutorial Task (强制 Retrieval Practice)

进入 tutorial 前, 你必须提交一篇 300 字 essay 回答以下三题 (Butler 2010 检索练习证据: 推断题 68% vs 重学 44%, 测试效应比反复阅读有效 24 个百分点):

1. 用 arxiv Python 包查询 "marketing analytics" 时, `arxiv.Search` 与 `Client().results()` 的调用顺序是什么? 网络异常 (HTTPError 限流) 时如何 fallback?
2. 给定一个 200 节点 3303 边的作者合作网络, 如何用 networkx 计算度中心性并识别 Top-5 核心作者? 边用 `itertools.combinations` 还是 `permutations`? 凭什么?
3. OSF 预注册如何对抗 p-hacking 与发表偏倚? FAIR 四字母分别指什么? ASReview 用什么机器学习范式加速文献综述?

**不交 essay 不进入 tutorial.** 这是 retrieval practice (提取练习), 不是 formative assessment——目的是让你在 tutorial 前先主动回忆, 暴露盲点.

Pre-task 提交后写入 `student_model.json` 的 `history` 字段, 作为 tutorial 起点的 mastery 基线.


In [ ]:
# Socratic Tutorial Loop (静态仿真, >=4 轮, >=5 苏格拉底问)
# 每轮检测学生 defense: 若失败则降一级 scaffold, 仍禁直接答案 (never give direct answers)

student_responses = [
    "我用 arxiv.Search(query='marketing analytics', max_results=50) 然后 Client().results(search) 拿 generator",
    "度中心性用 nx.degree_centrality(G), Top-5 用 sorted(..., key=lambda x: x[1], reverse=True)[:5]; 边用 combinations 去重",
    "OSF 预注册在数据收集前公开 hypothesis + analysis_plan, 这样就不能 p-hack; FAIR = Findable Accessible Interoperable Reusable",
    "ASReview 用主动学习 (Active Learning), 先让人标注少量, 模型挑最不确定的让人标",
    "一周内交文献综述: 先 arxiv 查询 -> ASReview 筛选 -> networkx 可视化 -> OSF 预注册",
]

# >=5 苏格拉底问 (禁直接答案, 每轮以追问结束)
socratic_questions = [
    # Turn 1 - arxiv API (为什么/若/依据)
    "为什么 Client().results() 返回 generator 而非 list? 若 arXiv API 限流返回 HTTPError, 你的 fallback 依据是什么? 反例: 若 query 为空会如何?",
    # Turn 2 - networkx (如何/若/凭什么/反例)
    "你的合作网络有 200 节点 3303 边, 如何确认边没有重复? 若两个作者合作 5 篇论文, 边权重应如何处理? 凭什么用度中心性而非介数中心性? 给一个反例 where 介数更优.",
    # Turn 3 - OSF/FAIR (若/如何/反例)
    "若研究者在预注册后又修改假设, OSF 如何发现? FAIR 的 'Interoperable' 在营销 A/B 测试数据中具体指什么? 反例: 一个 Findable 但不 Reusable 的数据集是什么样?",
    # Turn 4 - ASReview (若/假设.*变)
    "ASReview 的主动学习若初始标注有偏, 模型会放大偏差——你如何检测? 假设把 query 换成 'LLM marketing', ASReview 的 recall 会如何变? 为什么?",
    # Turn 5 - 综合 (如何/依据/若)
    "若你的导师要求一周内交营销 AI 文献综述, 你如何排列 arxiv 查询 -> ASReview 筛选 -> networkx 可视化 -> OSF 预注册的顺序? 依据是什么? 若 ASReview 中途崩溃, 你的 fallback 是什么?",
]

scaffold_level = 0  # 0=full Socratic, 1=hint, 2=worked example reference
turn_results = []

for turn, (resp, q) in enumerate(zip(student_responses, socratic_questions), 1):
    # 静态 if/else 模拟 defense 检测 (不调 LLM API)
    defense_ok = len(resp) > 15  # 简化: 学生有实质作答即部分 defense
    if not defense_ok:
        scaffold_level = min(scaffold_level + 1, 2)  # 降一级, 仍禁直接答案
        print(f"[Turn {turn}] defense 失败 -> scaffold_level={scaffold_level} (提示: 回顾 practice.md worked_faded 阶段1)")
    else:
        print(f"[Turn {turn}] defense 通过 -> 继续 Socratic 追问 (不降级)")
    print(f"  Fellow: {q}")
    print(f"  Student: {resp[:60]}...")
    print()
    turn_results.append({"turn": turn, "defense": "pass" if defense_ok else "fail", "scaffold": scaffold_level})

# 仍禁直接答案: 整个 loop 只提问, 不给答案
print(f"Tutorial 结束. scaffold_level={scaffold_level}. 共 {len(socratic_questions)} 轮 Socratic 追问.")
print(f"Turn results: {turn_results}")


In [ ]:
import json, os

# student_model.json 读写 (跨单元复用, 记录掌握度 / 盲点 / scaffold_level / history)
student_model = {
    "unit": "day-6-research-methodology",
    "topic": "研究方法论入门",
    "mastery": {
        "arxiv_api": 0.6,                # drill D1 掌握度 (ILO1)
        "pandas_bibliometrics": 0.5,     # drill D2 掌握度 (ILO2)
        "networkx_collab": 0.4,          # drill D2 掌握度 (ILO3, 200节点3303边)
        "reproducible_research": 0.3,    # drill D3 掌握度 (ILO4, OSF/Registered Reports)
        "fair_principles": 0.5,          # drill D3 掌握度 (ILO4)
        "asreview_active_learning": 0.2, # drill D3 掌握度 (ILO4, ASReview)
    },
    "blind_spots": [
        "OSF 预注册与 Registered Reports 的 Stage 1/Stage 2 区别",
        "FAIR 'Interoperable' 在企业数据治理中的具体落地与反例",
        "ASReview 初始标注偏差的检测方法 (Cohen's kappa)",
        "度中心性 vs 介数中心性的选择依据",
    ],
    "scaffold_level": 0,
    "last_updated": "2026-07-25",
    "history": [
        {"turn": 1, "topic": "arxiv API", "defense": "pass", "scaffold": 0},
        {"turn": 2, "topic": "networkx 合作网络", "defense": "pass", "scaffold": 0},
        {"turn": 3, "topic": "OSF/FAIR", "defense": "pass", "scaffold": 0},
        {"turn": 4, "topic": "ASReview", "defense": "pass", "scaffold": 0},
        {"turn": 5, "topic": "综合排序", "defense": "pass", "scaffold": 0},
    ],
    "review_queue": [
        "skill-0-day-1 (pandas 基础)",
        "schedule.json C4 (OSF 预注册)",
        "schedule.json C5 (FAIR/ASReview)",
        "skill-5-day-6 (IMRaD 写作)",
    ],
}

# 写入 (tutorial 结束后更新 mastery 与 blind_spots)
student_model_path = "student_model.json"
with open(student_model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

# 读取 (下次 tutorial 开始时加载, 跨单元复用)
if os.path.exists(student_model_path):
    with open(student_model_path, encoding="utf-8") as f:
        loaded = json.load(f)
    print(f"学生模型加载: {loaded['unit']} ({loaded['topic']})")
    print(f"掌握度: {loaded['mastery']}")
    print(f"盲点 ({len(loaded['blind_spots'])} 个): {loaded['blind_spots']}")
    print(f"复习队列: {loaded['review_queue']}")
    print(f"scaffold_level: {loaded['scaffold_level']}")


## Hattie (2007) Formative Feedback - 四级反馈

本 tutorial 的反馈按 Hattie 四级模型给出 (Hattie & Timperley, 2007, RER 77(1):81-112).
**避免 Self 级表扬** (如"你真聪明"), 聚焦 Task / Process / Self-Reg / Feed-Forward:

### [TASK] 任务级反馈 (关于具体任务的对错与质量)

- 你对 `arxiv.Search` 的调用顺序正确, 但未处理 `HTTPError` 限流场景——这是任务级缺陷.
- 你的 `nx.degree_centrality(G)` 调用正确, 但合作网络的边未做去重 (`itertools.combinations` 而非 `permutations`), 导致 3303 边中有重复.
- 你的 FAIR 四字母回答正确, 但 "Interoperable" 只写了"标准格式", 未具体到 Apache Arrow / JSON-LD.

### [PROCESS] 过程级反馈 (关于完成任务的策略与方法)

- 你选择度中心性而非介数中心性——对于"识别核心作者"这一任务, 度中心性更直接, 但你的论证缺失"为什么不用介数"的反例对比. 过程上, 应先写反例再下结论.
- 你的 OSF 预注册三件套 (hypothesis + analysis_plan + data_collection_order) 顺序正确, 但未说明 Registered Reports 的 Stage 1 (方法审稿) / Stage 2 (发表承诺) 区别——这是过程理解的不完整.
- 你对 ASReview "主动学习"的描述正确, 但未提及初始标注偏差的检测策略 (Cohen's kappa)——过程上应预设偏差检测.

### [SELF-REG] 自我调节级反馈 (关于学生对自己学习的监控)

- 你在 Turn 4 的 defense 暴露了"主动学习偏差检测"的盲点——建议自定步调 (self-paced) 回退到 drill D3 的 worked_faded 阶段 1, 24h 后重试. 这是自我调节: 识别盲点 -> 主动回退.
- 你能识别 FAIR 四字母, 但无法给出"Interoperable 在企业数据治理中的反例"——这是元认知缺口, 需用费曼话术自我解释一遍 (能否用 2 分钟向非技术同事讲清 Interoperable?).
- 你在 Turn 5 的综合排序缺少 fallback (若 ASReview 崩溃)——自我调节要求预设 Plan B.

### [FEED-FORWARD] 前馈级反馈 (指向下一个单元 / 下一次任务的可迁移能力)

- 下一步: 在项目 P5 (OSF 预注册包) 中, 补充一段 200 字的"若 hypothesis 被数据拒绝, 我的备选分析计划是什么"——这是 Feed-Forward, 指向技能5 Day 6 (IMRaD 写作) 的 Discussion 部分.
- 推荐复习: 技能0 Day 1 (pandas 基础) + 本单元 schedule.json C4/C5 卡片, 60 天内间隔重复 4 次 (FSRS-6, request_retention=0.9).
- 迁移: 把今天学到的"可复现研究三件套"迁移到技能3 (因果推断) 的 A/B 测试预注册——这是跨单元 Feed-Forward.


## 限频与 Exit Artifact

### 限频 (防 LLM 依赖, daily limit)

- **每单元每天 1 次**: 本 tutorial 每天最多进入 1 次 (daily limit = 1/day), 防止学生用 LLM 替代自己的 retrieval practice (提取练习). 限频是 Oxford tutorial 物理约束的仿真——真实 Oxford tutorial 每周 1 次, 强制学生在间隔期独立思考.
- **连续 2 次失败 -> weak_loop**: 触发弱项循环, 回退到上一 drill + worked_faded 阶段 1, 24h 后才能再次进入 tutorial.
- **使用日志**: 每次 tutorial 进入时间记录到 `student_model.json` 的 `history` 字段, 跨单元可追溯. 若发现某学生在 24h 内进入 >=2 次, 系统拒绝并提示"限频: 明天再来".
- **反依赖原则**: tutorial 不给直接答案 (never give direct answers), 只给 Socratic 追问. 若学生尝试通过反复进入 tutorial "套"答案, scaffold_level 不降反升 (增加独立性要求).

### Exit Artifact (出口交付物, 未交付 = tutorial 未完成)

完成本 tutorial 后, 你必须交付以下三项 (写入 `student_model.json`):

1. **2-3 个盲点** (写入 `blind_spots`):
   - 例: "OSF Registered Reports 的 Stage 1 审稿与 Stage 2 发表的承诺机制, 与普通预注册的区别"
   - 例: "ASReview 初始标注偏差如何用 Cohen's kappa 检测, 阈值多少算可接受"
   - 例: "度中心性 vs 介数中心性: 何时用介数? 给一个营销 AI 领域的反例"

2. **推荐复习单元** (写入 `review_queue`):
   - 例: 技能0 Day 1 (pandas 基础, groupby 复习)
   - 例: 本单元 schedule.json C4 (OSF 预注册, 60 天间隔重复)
   - 例: 技能5 Day 6 (IMRaD 写作, Feed-Forward 迁移目标)

3. **2 分钟话术** (口头, 不写入 json):
   - 口头回答 "营销 AI 领域 2026 年的研究热度与新兴方向是什么, 你的依据是什么"
   - 必须引用至少 1 篇 arxiv 查询到的真实论文 + 1 个合作网络发现的核心作者
   - Fellow 用 devil's advocate 角色追问 1 轮, 学生需 defense

**未交付 Exit Artifact = tutorial 未完成, 不进入下一单元.** 这是 Oxford tutorial 的出口关卡——真实 tutorial 也要求学生离开时带着"下一步要读什么"的清单.
